In [1]:
!pip install torch_geometric torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.9.0+cu126.html

Looking in links: https://data.pyg.org/whl/torch-2.9.0+cu126.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 66.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 65.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 95.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.6 MB/s eta 0:00:00


In [2]:
import argparse
import json
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
import pandas as pd
import numpy as np

from torch_geometric.loader import DataLoader
from torch_geometric.nn import GPSConv, GINConv, global_add_pool
from torch_geometric.transforms import AddLaplacianEigenvectorPE
from sklearn import metrics

In [3]:
class GPS(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, dropout, num_heads, pe_dim):
        """
        GPSConv model with LaplacianPE.

        pe_dim: number of Laplacian eigenvectors (fixed at PE_DIM=5).
        """
        super().__init__()
        self.dropout = dropout
        self.pe_dim = pe_dim

        # Projects the first pe_dim eigenvectors into hidden_dim
        self.pe_encoder = nn.Linear(pe_dim, hidden_dim)

        # Brings raw node features up to hidden_dim before GPS layers
        self.input_encoder = nn.Linear(input_dim, hidden_dim)

        # GPS layers: each wraps a local GINConv + global MultiheadAttention
        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            gin_mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(
                GPSConv(
                    channels=hidden_dim,
                    conv=GINConv(gin_mlp),
                    heads=num_heads,
                    dropout=dropout,
                    attn_type="multihead",
                )
            )

        self.lin1 = nn.Linear(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        pe = data.laplacian_eigenvector_pe
        x = self.input_encoder(x) + self.pe_encoder(pe)

        # GPS layers with jumping-knowledge sum over layers
        h_graph = 0
        for conv in self.convs:
            x = conv(x, edge_index, batch)
            x = F.dropout(x, self.dropout, training=self.training)
            h_graph = h_graph + global_add_pool(x, batch)

        h = F.relu(self.lin1(h_graph))
        h = F.dropout(h, self.dropout, training=self.training)
        return self.classifier(h).view(-1)

In [4]:
def train(model, optimizer, criterion, train_loader, device):
    model.train()
    total_loss = 0

    for data in train_loader:
        data = data.to(device)
        out = model(data)
        loss = criterion(out, data.y.float())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * data.num_graphs
        
    return total_loss / len(train_loader.dataset)


@torch.no_grad()
def test(model, loader, device):
    model.eval()
    predictions = []
    labels = []

    for data in loader:
        data = data.to(device)
        out = model(data)
        pred = (out > 0).float()
        predictions.append(pred.cpu())
        labels.append(data.y.cpu())

    accuracy = metrics.accuracy_score(torch.cat(labels), torch.cat(predictions))
    f1 = metrics.f1_score(torch.cat(labels), torch.cat(predictions))

    return accuracy, f1

In [5]:
parser = argparse.ArgumentParser(description="GIN for partial automorphism extension problem")
parser.add_argument("--seed", type=int, default=43,
                    help="Random seed for reproducibility (default: 43)") 
parser.add_argument("--config", type=str, default="/kaggle/input/datasets/samuelvarchol/graphs-with-autgrpg-1/best_config_gps.json",
                    help="Use config file for hyperparameters")
parser.add_argument("--hidden_dim", type=int,
                    help="Hidden dimension size") 
parser.add_argument("--num_layers", type=int, 
                    help="Number of GIN layers")
parser.add_argument("--dropout", type=float, 
                    help="Dropout rate")
parser.add_argument("--lr", type=float,
                    help="Learning rate")
parser.add_argument("--weight_decay", type=float,
                    help="Weight decay")
parser.add_argument("--batch_size", type=int,
                    help="Input batch size")
parser.add_argument("--num_heads", type=int,
                    help="Number of attention heads")
parser.add_argument("--factor", type=float, default=0.5,
                    help="Factor for learning rate scheduler (default: 0.5)")
parser.add_argument("--patience", type=int, default=3,
                    help="Patience for learning rate scheduler (default: 3)")
parser.add_argument("--epochs", type=int, default=150,
                    help="Number of epochs to train")

args, remaining_argv = parser.parse_known_args()

with open(args.config, "r") as f:
    config_dict = json.load(f)
parser.set_defaults(**config_dict)

args = parser.parse_args("")

In [6]:
PE_DIM = 5
pe_transform = AddLaplacianEigenvectorPE(
    k=PE_DIM,
    attr_name="laplacian_eigenvector_pe",
    is_undirected=True,
)

train_dataset_raw = torch.load(
    "/kaggle/input/datasets/samuelvarchol/graphs-with-autgrpg-1/train_dataset_baseline.pt",
    weights_only=False,
)
val_dataset_raw = torch.load(
    "/kaggle/input/datasets/samuelvarchol/graphs-with-autgrpg-1/val_dataset.pt",
    weights_only=False,
)

print("Applying LaplacianPE transform to train set...")
train_dataset = [pe_transform(data) for data in train_dataset_raw]
print("Applying LaplacianPE transform to val set...")
val_dataset   = [pe_transform(data) for data in val_dataset_raw]
print(f"Done. PE shape per node: {train_dataset[0].laplacian_eigenvector_pe.shape}  (fixed pe_dim={PE_DIM})")

number_of_features = train_dataset[0].num_node_features
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

Applying LaplacianPE transform to train set...
Applying LaplacianPE transform to val set...
Done. PE shape per node: torch.Size([13, 5])  (fixed pe_dim=5)


In [7]:
# ── Seeds used across runs (standard practice in GNN papers) ──────────────────
SEEDS = [0, 1, 2, 3, 4]

def run_single_seed(seed):
    """Full training pipeline for a single seed. Returns best val_acc, val_f1, and history."""
    torch_geometric.seed_everything(seed)

    train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset,   batch_size=args.batch_size)

    model = GPS(number_of_features, args.hidden_dim, args.num_layers, args.dropout, args.num_heads, PE_DIM).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    criterion = nn.BCEWithLogitsLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=args.factor, patience=args.patience
    )

    best_val_acc, best_val_f1 = 0.0, 0.0
    patience_counter = 0
    training_history = []
    best_state_dict = None

    for epoch in range(1, args.epochs + 1):
        train_loss = train(model, optimizer, criterion, train_loader, device)
        train_acc, _ = test(model, train_loader, device)
        val_acc, val_f1 = test(model, val_loader, device)
        scheduler.step(val_acc)

        training_history.append({
            "seed": seed, "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_acc": val_acc,   "val_f1":   val_f1,
            "lr": optimizer.param_groups[0]["lr"],
        })

        if val_acc > best_val_acc:
            best_val_acc, best_val_f1 = val_acc, val_f1
            patience_counter = 0
            best_state_dict = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1

        if patience_counter >= 15:
            print(f"  [seed {seed}] Early stopping at epoch {epoch}.")
            break

        if epoch % 10 == 0:
            print(f"  [seed {seed}] Epoch {epoch:03d} | "
                  f"loss {train_loss:.4f} | "
                  f"train acc {train_acc:.4f} | "
                  f"val acc {val_acc:.4f} | val f1 {val_f1:.4f}")

    return best_val_acc, best_val_f1, training_history, best_state_dict


# ── Run all seeds ─────────────────────────────────────────────────────────────
all_val_acc, all_val_f1 = [], []
all_results = []  # (seed, val_acc, val_f1, history, state_dict)

for seed in SEEDS:
    print(f"\n{'='*55}\nRun seed={seed}")
    val_acc, val_f1, history, state_dict = run_single_seed(seed)
    all_val_acc.append(val_acc)
    all_val_f1.append(val_f1)
    all_results.append((seed, val_acc, val_f1, history, state_dict))
    print(f"  → best val acc: {val_acc:.4f}  |  best val f1: {val_f1:.4f}")

# ── Aggregate results (paper-style reporting) ─────────────────────────────────
acc_arr = np.array(all_val_acc)
f1_arr  = np.array(all_val_f1)

print(f"\n{'='*55}")
print("Multi-seed results (mean ± std over 5 runs):")
print(f"  Val Accuracy : {acc_arr.mean():.4f} ± {acc_arr.std():.4f}")
print(f"  Val F1 Score : {f1_arr.mean():.4f}  ± {f1_arr.std():.4f}")
print(f"{'='*55}")
print("Per-seed breakdown:")
for seed, acc, f1, _, __ in all_results:
    print(f"  Seed {seed}: acc={acc:.4f}  f1={f1:.4f}")

# ── Save only the global best model's checkpoint and history ──────────────────
best_seed, best_acc, best_f1, best_history, best_state = max(all_results, key=lambda x: x[1])

torch.save(best_state, "/kaggle/working/best_model.pt")
pd.DataFrame(best_history).to_csv("/kaggle/working/best_model_history.csv", index=False)

print(f"\nSaved checkpoint + history for seed={best_seed} (val_acc={best_acc:.4f})")

# ── Save summary table for all seeds ─────────────────────────────────────────
summary_df = pd.DataFrame({
    "seed":    SEEDS,
    "val_acc": all_val_acc,
    "val_f1":  all_val_f1,
})
summary_df.loc[len(summary_df)] = ["mean", acc_arr.mean(), f1_arr.mean()]
summary_df.loc[len(summary_df)] = ["std",  acc_arr.std(),  f1_arr.std()]
summary_df.to_csv("/kaggle/working/multi_seed_summary.csv", index=False)
print("Saved: multi_seed_summary.csv")


Run seed=0
  [seed 0] Epoch 010 | loss 0.2468 | train acc 0.9078 | val acc 0.9011 | val f1 0.9093
  [seed 0] Epoch 020 | loss 0.1603 | train acc 0.9516 | val acc 0.9329 | val f1 0.9376
  [seed 0] Epoch 030 | loss 0.0975 | train acc 0.9723 | val acc 0.9486 | val f1 0.9524
  [seed 0] Epoch 040 | loss 0.0614 | train acc 0.9888 | val acc 0.9571 | val f1 0.9602
  [seed 0] Epoch 050 | loss 0.0436 | train acc 0.9950 | val acc 0.9548 | val f1 0.9581
  [seed 0] Early stopping at epoch 55.
  → best val acc: 0.9571  |  best val f1: 0.9602

Run seed=1
  [seed 1] Epoch 010 | loss 0.2344 | train acc 0.9116 | val acc 0.8969 | val f1 0.8995
  [seed 1] Epoch 020 | loss 0.1496 | train acc 0.9489 | val acc 0.9329 | val f1 0.9393
  [seed 1] Epoch 030 | loss 0.1156 | train acc 0.9664 | val acc 0.9443 | val f1 0.9490
  [seed 1] Epoch 040 | loss 0.0947 | train acc 0.9754 | val acc 0.9493 | val f1 0.9533
  [seed 1] Epoch 050 | loss 0.0519 | train acc 0.9916 | val acc 0.9592 | val f1 0.9622
  [seed 1] Epoch 0